# Structured Output (Basic)

| Concept | Purpose |
| --- | --- |
| **Free-text output** | Normal LLM reply (just a string), hard to use in code |
| **Pydantic BaseModel** | Define the shape (fields + types) we want |
| **Field(description=...)** | Tell the LLM what each field means |
| **with_structured_output()** | Make the LLM return data in that shape |
| **List of models** | Get many items (like 3 movies) in one response |

LLM normally gives **text**. Structured output makes it give **data** (name, email, age as proper fields) so we can use it directly in code.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## 1. The Problem

**Short note:** We have messy text and want name, email, age out of it. The LLM replies with a normal sentence (a string). We can't safely do `res.name` or `res.age` on a string.

In [ ]:
text = "My name is Muktananda Hiremath. Email is test@gmail.com and age is 21"

res = llm.invoke(f"Give me the name, email and age from this text: {text}")

print(res.content)
print(type(res.content))   # str, just text

## 2. Pydantic BaseModel + Field

**Short note:** Pydantic lets us define **what the output should look like**, like a form with fields.

- `BaseModel` → the class that holds the fields
- `name: str` → field name and its type
- `Field(description=...)` → tells the LLM what to put in that field

In [ ]:
from pydantic import BaseModel, Field

class UserInfo(BaseModel):
    name: str  = Field(description="Name of the user")
    email: str = Field(description="Email of the user")
    age: int   = Field(description="Age of the user")

## 3. with_structured_output()

**Short note:** Give the model our class. It now returns an **object of that class** instead of text.

In [ ]:
structured_llm = llm.with_structured_output(UserInfo)

res = structured_llm.invoke(f"Extract the user info from this text: {text}")

print(res)
print(type(res))

In [ ]:
# now we can use it like normal Python data
print(res.name)
print(res.email)
print(res.age + 1)   # age is a real int, so maths works

In [ ]:
# convert to dictionary / JSON if needed
print(res.model_dump())

## 4. One Item vs Many Items

**Short note:** A model describes **one** item. If we want many (like 3 movies), wrap it in a `List`.

Common mistake: `movies: Movies` gives only one. Use `movies: List[Movies]` for many.

In [ ]:
from typing import List

class Movie(BaseModel):
    name: str = Field(description="Movie title")
    year: int = Field(description="Release year")

class AllMovies(BaseModel):
    movies: List[Movie] = Field(description="List of movies")

movie_llm = llm.with_structured_output(AllMovies)

res = movie_llm.invoke("Give me 3 recent Kannada movies")

for m in res.movies:
    print(m.name, "-", m.year)

## Quick Summary

| Concept | One line | Code idea |
| --- | --- | --- |
| Free-text output | LLM gives a plain string | `res.content` |
| Pydantic BaseModel | define the shape we want | `class UserInfo(BaseModel)` |
| Field | explain each field to the LLM | `Field(description="...")` |
| with_structured_output | LLM returns our class | `llm.with_structured_output(UserInfo)` |
| List of models | many items at once | `movies: List[Movie]` |

**Common mistakes**
- Package name is `pydantic` (not `pydentic`)
- Use `Field(...)`, not some other class
- Method is `invoke` (not `inovoke`)
- Use `List[Movie]` when you want many items